In [ ]:
from pathlib import Path
from pprint import pformat, pprint

import os
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_ninapro_csv

In [ ]:
csv_path = Path("data/raw/Ninapro_DB1.csv")
figure_dir = Path("reports/figures")
figure_dir.mkdir(parents = True, exist_ok = True)

sampling_rate_hz = 100

approved_missing_values: dict[str, int] = {}

# Load the real dataset and capture the full loader report
df, report = load_ninapro_csv(csv_path, exercises = (2, 3))

pprint(report, sort_dicts = False, width = 120)

(Path("reports") / "eda_loader_report.txt").write_text(
    pformat(report, sort_dicts = False, width = 120),
    encoding = "utf-8",
)

emg_columns = report["emg_columns"]
if report["has_ten_emg_columns"] is not True:
    raise RuntimeError(
        f"Expected 10 EMG columns but detected {report['emg_column_count']}: {emg_columns}. "
        "Inspect the downloaded CSV schema before continuing."
    )

In [ ]:
# Cross check expected DB1 characteristics before EDA

active = df.loc[df["restimulus"].ne(0)].copy()

verified = {
    "participants_in_full_csv": len(report["unique_subject_ids"]),
    "selected_exercises": report["selected_exercises"],
    "sampling_rate_hz_assumed_from_DB1_documentation": sampling_rate_hz,
    "emg_column_count": report["emg_column_count"],
    "exercise_b_active_classes": int(
        active.loc[active["exercise"].eq(2), "restimulus"].nunique()
    ),
    "exercise_c_active_classes": int(
        active.loc[active["exercise"].eq(3), "restimulus"].nunique()
    ),
    "rest_present": bool(df["restimulus"].eq(0).any()),
    "raw_rows_before_filtering": report["row_count_before_filtering"],
    "rows_after_exercise_b_c_filter": report["row_count_after_filtering"],
    "missing_value_counts": report["missing_value_counts"],
    "label_scope_check": report["restimulus_label_scope"],
}

print("\nVerified DB1 characteristics:")
pprint(verified, sort_dicts = False, width = 120)

expected = {
    "participants_in_full_csv": 27,
    "exercise_b_active_classes": 17,
    "exercise_c_active_classes": 23,
    "emg_column_count": 10,
    "rest_present": True,
}

mismatches = {
    name: {"expected": expected_value, "found": verified[name]}
    for name, expected_value in expected.items()
    if verified[name] != expected_value
}
if mismatches:
    raise RuntimeError(
        "DB1 validation mismatch. Investigate this before continuing:\n"
        f"{pformat(mismatches, sort_dicts = False)}"
    )

unexpected_missing = {
    column: count
    for column, count in report["missing_value_counts"].items()
    if count > approved_missing_values.get(column, 0)
}
if unexpected_missing:
    raise RuntimeError(
        "Unexpected missing values found. Inspect and explain them before EDA:\n"
        f"{pformat(unexpected_missing, sort_dicts = False)}"
    )

In [ ]:
# EDA-only labels
# `eda_combined_label` distinguishes every (exercise, restimulus) pair, including exercise specific rest labels. It proves collisions are resolved.
# `eda_plot_label` unifies rest into one class so the balance chart reflects the intended 41 class project scope: 17 + 23 + one rest class.

df = df.reset_index(names = "source_row")

df["eda_combined_label"] = (
    "E" + df["exercise"].astype(str) + "_L" + df["restimulus"].astype(str)
)

df["eda_plot_label"] = np.where(
    df["restimulus"].eq(0),
    "rest",
    df["eda_combined_label"],
)

active_label_collisions = (
    active.groupby("restimulus")["exercise"]
    .nunique()
    .loc[lambda values: values.gt(1)]
)

for raw_label in active_label_collisions.index:
    resolved_count = (
        df.loc[df["restimulus"].eq(raw_label), "eda_combined_label"].nunique()
    )
    assert resolved_count >= 2, (
        f"Raw restimulus {raw_label} was not separated by exercise."
    )

print(f"\nEDA combined label count: {df['eda_combined_label'].nunique()}")
print(f"EDA plotting label count: {df['eda_plot_label'].nunique()}")

if df["eda_plot_label"].nunique() != 41:
    raise RuntimeError(
        "Expected 41 EDA plotting classes. Investigate missing or unexpected "
        "Exercise B/C/rest labels before continuing."
    )

In [ ]:
# Timing and segment-quality screening.
# A contiguous labeled run establishes label presence, not signal quality.
# DB1's refined restimulus boundaries are variable, so do not impose an unsupported fixed duration pass/fail rule.
# Convert samples to seconds at the documented 100 Hz rate and inspect the empirical distribution instead.

run_keys = ["subject", "exercise", "restimulus", "rerepetition"]
df["run_id"] = (
    df[run_keys]
    .ne(df[run_keys].shift())
    .any(axis = 1)
    .cumsum()
)

run_summary = (
    df.groupby(["run_id", "subject", "exercise", "restimulus", "rerepetition"])
    .agg(
        n_samples = ("source_row", "size"),
        first_source_row = ("source_row", "min"),
        last_source_row = ("source_row", "max"),
    )
    .reset_index()
)
run_summary["duration_seconds"] = run_summary["n_samples"] / sampling_rate_hz
run_summary["segment_type"] = np.where(
    run_summary["restimulus"].eq(0), "rest", "active"
)

active_runs = run_summary.loc[run_summary["segment_type"].eq("active")].copy()
rest_runs = run_summary.loc[run_summary["segment_type"].eq("rest")].copy()

timing_summary = (
    run_summary.groupby("segment_type")["n_samples"]
    .agg(["count", "min", "median", "mean", "max", "std"])
    .round(3)
)
timing_summary["duration_min_s"] = timing_summary["min"] / sampling_rate_hz
timing_summary["duration_median_s"] = timing_summary["median"] / sampling_rate_hz
timing_summary["duration_mean_s"] = timing_summary["mean"] / sampling_rate_hz
timing_summary["duration_max_s"] = timing_summary["max"] / sampling_rate_hz

print("Sample-count and duration summary at 100 Hz:")
print(timing_summary.to_string())

# This is a descriptive screening threshold, not a diagnosis of corruption.
active_outside_2_to_6_s = active_runs.loc[
    ~active_runs["duration_seconds"].between(2, 6)
]
print(
    f"\nActive runs outside 2–6 s: {len(active_outside_2_to_6_s):,} "
    f"of {len(active_runs):,} ({len(active_outside_2_to_6_s) / len(active_runs):.2%})"
)

# There is no timestamp column, so exact sample drop/duplication detection is not possible.
# This check confirms no gaps inside the filtered table runs.
assert (
    run_summary["last_source_row"] - run_summary["first_source_row"] + 1
).eq(run_summary["n_samples"]).all()

run_summary.to_csv("reports/eda_segment_timing.csv", index=False)

In [ ]:
# Figure: one rest segment and one active segment, all 10 EMG channels

def longest_segment(segment_type: str) -> int:
    return int(
        run_summary.loc[run_summary["segment_type"].eq(segment_type)]
        .sort_values("n_samples", ascending = False)
        .iloc[0]["run_id"]
    )

rest_run_id = longest_segment("rest")
active_run_id = longest_segment("active")

rest_segment = df.loc[df["run_id"].eq(rest_run_id)]
active_segment = df.loc[df["run_id"].eq(active_run_id)]

fig, axes = plt.subplots(2, 1, figsize = (14, 9), sharex = False)

for axis, segment, title in [
    (axes[0], rest_segment, "Rest segment (Longest observed)"),
    (axes[1], active_segment, "Active gesture segment"),
]:
    time_seconds = np.arange(len(segment)) / sampling_rate_hz

    for channel in emg_columns:
        axis.plot(
            time_seconds,
            segment[channel].to_numpy(),
            linewidth = 0.7,
            alpha = 0.8,
            label = channel,
        )

    metadata = segment.iloc[0]
    axis.set_title(
        f"{title}: subject = {metadata['subject']}, "
        f"exercise = {metadata['exercise']}, "
        f"restimulus = {metadata['restimulus']}, "
        f"rerepetition = {metadata['rerepetition']}"
    )
    axis.set_xlabel("Time (s)")
    axis.set_ylabel("Raw EMG amplitude")
    axis.grid(alpha = 0.25)
    axis.legend(
        title = "EMG channel",
        ncol = 5,
        fontsize = 8,
        loc = "upper center",
        bbox_to_anchor = (0.5, -0.18),
    )

fig.suptitle("Raw multi-channel sEMG examples", y = 1.02, fontsize = 15)
fig.tight_layout()
fig.savefig(figure_dir / "raw_multichannel_trace.png", dpi = 300, bbox_inches = "tight")

plt.show()

In [ ]:
# Figure: class balance across the intended 41 plotting classes

class_balance = (
    df["eda_plot_label"]
    .value_counts()
    .rename_axis("class_label")
    .reset_index(name = "sample_count")
)

rest_samples = int(class_balance.loc[class_balance["class_label"].eq("rest"), "sample_count"].iloc[0])
active_samples = int(class_balance.loc[~class_balance["class_label"].eq("rest"), "sample_count"].sum())
total_samples = rest_samples + active_samples
active_counts = class_balance.loc[~class_balance["class_label"].eq("rest"), "sample_count"]

balance_summary = {
    "rest_samples": rest_samples,
    "active_samples": active_samples,
    "total_samples": total_samples,
    "rest_percent": rest_samples / total_samples * 100,
    "active_percent": active_samples / total_samples * 100,
    "rest_to_active_ratio": rest_samples / active_samples,
    "active_label_min": int(active_counts.min()),
    "active_label_max": int(active_counts.max()),
    "active_label_cv": float(active_counts.std() / active_counts.mean()),
}
print("Rest-versus-active balance:")
pprint(balance_summary, sort_dicts=False)

plt.figure(figsize = (16, 7))

sns.barplot(
    data = class_balance,
    x = "class_label",
    y = "sample_count",
    color = "#2A6FBB"
)

plt.title("Class balance across Exercise B, Exercise C, and unified rest")
plt.xlabel("EDA class label")
plt.ylabel("Samples")
plt.xticks(rotation = 90)
plt.tight_layout()
plt.savefig(figure_dir / "class_balance.png", dpi = 300, bbox_inches = "tight")

plt.show()

class_balance.to_csv("reports/eda_class_balance.csv", index = False)

In [ ]:
# Figure: active repetition presence per participant
# 17 Exercise B + 23 Exercise C movements = 40 active movements.
# At 10 repetitions each, the nominal count is 400 active labeled runs/person.
# This validates label presence only; use the timing cell above for duration quality.

active_runs["eda_plot_label"] = (
    "E" + active_runs["exercise"].astype(str) + "_L" + active_runs["restimulus"].astype(str)
)

reps_per_subject = (
    active_runs.groupby("subject")
    .size()
    .rename("completed_active_repetitions")
    .reset_index()
    .sort_values("subject")
)

expected_active_reps_per_subject = (17 + 23) * 10
reps_per_subject["difference_from_nominal_400"] = (
    reps_per_subject["completed_active_repetitions"] - expected_active_reps_per_subject
)
repetition_count_summary = reps_per_subject["completed_active_repetitions"].agg(
    ["min", "max", "mean", "std"]
)

print("Repetition counts per subject:")
print(reps_per_subject.to_string(index = False))
print("\nCount summary:")
print(repetition_count_summary.to_string())

assert reps_per_subject["completed_active_repetitions"].eq(expected_active_reps_per_subject).all()

plt.figure(figsize = (14, 6))

sns.barplot(
    data = reps_per_subject,
    x = "subject",
    y = "completed_active_repetitions",
    color = "#3C8D40"
)

plt.axhline(
    expected_active_reps_per_subject,
    color = "#B22222",
    linestyle = "--",
    label = "Nominal expectation: 400 active reps",
)

plt.title("Active labeled repetitions per subject")
plt.xlabel("Subject ID")
plt.ylabel("Number of contiguous active labeled runs")
plt.legend()
plt.tight_layout()
plt.savefig(figure_dir / "repetitions_per_subject.png", dpi = 300, bbox_inches = "tight")

plt.show()

reps_per_subject.to_csv("reports/eda_repetitions_per_subject.csv", index = False)

## Rest-labeled transient-activity screening

The 109 second rest example is plausible because `rerepetition == 0` includes unlabeled/baseline periods. However, high amplitude events inside rest labeled runs are a **dataset-wide pattern**, not a one-off: the prior full data scan found 1,939 of 10,854 rest runs (17.86%) with a peak absolute amplitude at or above 1.0, and 803 (7.40%) at or above 1.4. All 27 subjects appear above both thresholds.

This is a descriptive screen, not proof of artefact or a basis to delete data.

In [ ]:
# Quantify peak absolute amplitude in each rest-labeled contiguous run.
# Aggregate channel minima/maxima first to avoid constructing a large row-wise array.

rest_channel_extrema = (
    df.loc[df["restimulus"].eq(0)]
    .groupby("run_id")[emg_columns]
    .agg(["min", "max"])
)
rest_peak_by_run = rest_channel_extrema.abs().max(axis = 1).rename("peak_abs_amplitude")

rest_peak_summary = (
    rest_runs.merge(rest_peak_by_run, on = "run_id", how = "left")
    .sort_values("peak_abs_amplitude", ascending = False)
)

peak_descriptives = rest_peak_summary["peak_abs_amplitude"].describe(
    percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
print("Peak absolute amplitude per rest-labeled run:")
print(peak_descriptives.round(4).to_string())

for threshold in (1.0, 1.4, 2.0):
    flagged = rest_peak_summary.loc[
        rest_peak_summary["peak_abs_amplitude"].ge(threshold)
    ]
    print(
        f"Peak ≥ {threshold:.1f}: {len(flagged):,}/{len(rest_peak_summary):,} "
        f"({len(flagged) / len(rest_peak_summary):.2%}); "
        f"subjects represented: {flagged['subject'].nunique()}"
    )

rest_peak_summary.to_csv("reports/eda_rest_peak_by_segment.csv", index = False)

plt.figure(figsize=(10, 6))

sns.histplot(rest_peak_summary["peak_abs_amplitude"],
             bins = 80,
             color = "#7B4EA3"
)

plt.axvline(1.0,
            color = "#B22222",
            linestyle = "--",
            label = "Screening threshold: 1.0"
)

plt.axvline(1.4,
            color = "#E67E22",
            linestyle = "--",
            label = "Screening threshold: 1.4"
)

plt.title("Peak absolute EMG amplitude across rest-labeled segments")
plt.xlabel("Maximum absolute amplitude within segment")
plt.ylabel("Rest-labeled segments")
plt.legend()
plt.tight_layout()
plt.savefig(figure_dir / "rest_peak_distribution.png", dpi = 300, bbox_inches = "tight")

plt.show()

## EDA summary

The complete mirror contains 27 subjects, 10 EMG channels, no missing values, and the expected 17 Exercise B plus 23 Exercise C active classes. The 41-class EDA view has 5,748,692 rest samples (58.53%) and 4,073,526 active samples (41.47%); active class counts are relatively uniform. Each subject has all 400 expected active labeled runs, but this is a label completeness result rather than a quality guarantee. Active run durations range from 1.84 to 6.61 seconds at 100 Hz. Rest labeled high amplitude events occur across all subjects and will be treated as a documented preprocessing risk, not silently removed.

In [ ]:
# Output values to paste into DATASET_MANIFEST.md
# Compute the rate of adjacent rows whose 10 EMG channels are identical.
# This is a row-to-row duplicate check on the filtered dataset.

duplicate_adjacent_rate = (
    df[emg_columns]
    .eq(df[emg_columns].shift(1))
    .all(axis=1)
    .mean()
)

manifest_values = {
    "Participants": verified["participants_in_full_csv"],
    "Acquisition rate": f"{sampling_rate_hz} Hz (DB1 documented rate)",
    "sEMG channels": verified["emg_column_count"],
    "Exercise B classes": verified["exercise_b_active_classes"],
    "Exercise C classes": verified["exercise_c_active_classes"],
    "Project target classes": df["eda_plot_label"].nunique(),
    "Columns": ", ".join(report["columns"]),
    "EMG columns": ", ".join(emg_columns),
    "Row count": report["row_count_before_filtering"],
    "Rows after B/C filter": report["row_count_after_filtering"],
    "Missing values": report["missing_value_counts"],
    "Raw-label collision result": report["restimulus_label_scope"],
    "Timing summary": timing_summary.to_dict(),
    "Adjacent identical EMG-row rate": f"{duplicate_adjacent_rate:.6%}",
}

print("\nCopy these verified values into DATASET_MANIFEST.md:")
pprint(manifest_values, sort_dicts = False, width = 120)